# 05b — Canastas GBA retropoladas 2003-marzo 2016 (Tornarolli 2018, CEDLAS DT 226)

Construye la canasta GBA (CBA/CBT por adulto equivalente) mensual y trimestral para
el período histórico, con el procedimiento de **dos anclas** de Tornarolli (2018):
diciembre 2006 (Metodología INDEC N°22) y abril 2016 (canasta oficial), un índice
encadenado de IPC, extendida a las 6 regiones vía la proporción regional oficial de
abril 2016.

**No calcula distancia por hogar ni fija ninguna tasa de pobreza/indigencia propia**
-- las tasas semestrales de la última sección son solo un chequeo de contraste contra
Tornarolli/CEDLAS, a revisar antes de decidir el próximo paso.

**Lee** `data/insumos_lineas/`, `data/canastas_indec/`, `data/02_nucleo/{hogar,individual}`,
`data/01_armonizado/hogar`. **Escribe** `data/meta/canasta_retropolada_*.csv`.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import pipeline_meta as pm
from armonizacion import trimestres_publicados
import canasta_regional
import insumos_lineas as il
import distancia_linea_pobreza
import canasta_retropolada as cr
import ponderadores

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
META = REPO / "data" / "meta"
CANASTAS = REPO / "data" / "canastas_indec"
INSUMOS = REPO / "data" / "insumos_lineas"
RETROPOLACION = CANASTAS / "retropolacion"

TODOS_LOS_TRIMESTRES = trimestres_publicados()
TRIMESTRES_HISTORICOS = [t for t in TODOS_LOS_TRIMESTRES if t < (2016, 2)]
print(f"{len(TRIMESTRES_HISTORICOS)} trimestres históricos disponibles "
      f"({TRIMESTRES_HISTORICOS[0]}-{TRIMESTRES_HISTORICOS[-1]})")

## Anclas

Diciembre 2006 (Metodología INDEC N°22) y abril 2016 (canasta oficial, 6 regiones).

In [ ]:
ancla_dic_2006 = cr.cargar_ancla_dic_2006(INSUMOS / "dic_2006_gba_metodologia22.csv")
ancla_abr_2016 = cr.cargar_ancla_abr_2016(CANASTAS)
print(f"ancla dic-2006 (GBA): {ancla_dic_2006}")
display(ancla_abr_2016)

## Índice encadenado P(m)

IPC-GBA INDEC hasta diciembre 2006; desde enero 2007, IPC Provincias CIFRA
reescalado para empalmar con el nivel INDEC de ese mes. La variación en el
empalme es un dato ya bajo intervención INDEC -- se registra el valor, no se
corrige.

In [ ]:
indec_largo = il.cargar_ipc_gba_historico(INSUMOS / "ipc_gba_historico_1943_2008_crudo.csv")
indec_nivel_general = indec_largo[indec_largo["serie"] == "ipc_gba_nivel_general"].set_index("mes")["valor"]
cifra_nivel_general = il.cargar_ipc_provincias_cifra(
    INSUMOS / "ipc_provincias_cifra_2007_2018_crudo.xlsx").set_index("mes")["valor"]

indice, variacion_empalme_ene_2007 = cr.encadenar_indice(indec_nivel_general, cifra_nivel_general)
print(f"variación IPC-GBA INDEC dic-2006 -> ene-2007: {variacion_empalme_ene_2007:+.2%} "
      "(primer mes bajo intervención INDEC -- se registra, no se corrige)")
print(f"índice dic-2006: {indice[pd.Timestamp('2006-12-01')]:.2f}")
print(f"índice abr-2016: {indice[pd.Timestamp('2016-04-01')]:.2f}")

## Canasta GBA mensual, dos anclas (más la versión de un solo ancla, sensibilidad)

`construir_canasta_gba_mensual` arma el tramo temprano con el ancla de diciembre-2006
y el tramo posterior con el promedio ponderado de las dos anclas -- agrega además la
versión de un solo ancla para la validación de sensibilidad.

In [ ]:
cba_gba_mensual = cr.construir_canasta_gba_mensual(
    ancla_dic_2006["cba_adulto_equivalente"], ancla_abr_2016.set_index("REGION").loc[cr.REGION_GBA, "cba_adulto_equivalente"],
    indice, "cba_adulto_equivalente")
cbt_gba_mensual = cr.construir_canasta_gba_mensual(
    ancla_dic_2006["cbt_adulto_equivalente"], ancla_abr_2016.set_index("REGION").loc[cr.REGION_GBA, "cbt_adulto_equivalente"],
    indice, "cbt_adulto_equivalente")
print(f"{len(cba_gba_mensual)} filas (cba_adulto_equivalente), {len(cbt_gba_mensual)} filas (cbt_adulto_equivalente)")
display(cba_gba_mensual[cba_gba_mensual["origen"] == cr.ORIGEN_DOS_ANCLAS].head(3))
display(cba_gba_mensual[cba_gba_mensual["origen"] == cr.ORIGEN_UNA_ANCLA].head(3))

## Extensión a las 6 regiones y agregación trimestral

`extender_regiones` escala la serie GBA por la proporción oficial de cada región;
`agregar_trimestral_por_origen` promedia los meses de cada trimestre, separado por
`origen`.

In [ ]:
cba_regiones_mensual = cr.extender_regiones(cba_gba_mensual, ancla_abr_2016, "cba_adulto_equivalente")
cbt_regiones_mensual = cr.extender_regiones(cbt_gba_mensual, ancla_abr_2016, "cbt_adulto_equivalente")

cba_regiones_trimestral = cr.agregar_trimestral_por_origen(cba_regiones_mensual, "cba_adulto_equivalente")
cbt_regiones_trimestral = cr.agregar_trimestral_por_origen(cbt_regiones_mensual, "cbt_adulto_equivalente")

canasta_trimestral_retropolada = cba_regiones_trimestral.merge(
    cbt_regiones_trimestral, on=["ANO4", "TRIMESTRE", "REGION", "origen"], suffixes=("", "_cbt"))
print(f"{len(canasta_trimestral_retropolada)} filas región×trimestre×origen")
display(canasta_trimestral_retropolada.head(3))

## Ensamblado final: retropolada + oficial, formato tabular común

La canasta oficial se arma con `canasta_regional`, ya existente -- se concatena con
la retropolada en una sola tabla larga y se escribe a `data/meta/`.

In [ ]:
oficial_mensual_cba = canasta_regional.cargar_canasta_larga(CANASTAS / "cba_regional_crudo.csv", "cba_adulto_equivalente")
oficial_mensual_cbt = canasta_regional.cargar_canasta_larga(CANASTAS / "cbt_regional_crudo.csv", "cbt_adulto_equivalente")
oficial_trimestral = canasta_regional.construir_canasta_trimestral(CANASTAS, "a_mismo_trimestre")


def _mensual_retropolada_larga(df, variable):
    salida = df.rename(columns={variable: "valor"})[["REGION", "ANO4", "MES", "valor", "origen"]].copy()
    salida["serie"] = variable
    return salida


def _mensual_oficial_larga(df, variable):
    salida = df.rename(columns={variable: "valor"})[["REGION", "ANO4", "MES", "valor"]].copy()
    salida["serie"] = variable
    salida["origen"] = cr.ORIGEN_OFICIAL
    return salida


def _trimestral_retropolada_larga(df, variable):
    salida = df.rename(columns={variable: "valor"})[["REGION", "ANO4", "TRIMESTRE", "valor", "origen"]].copy()
    salida["serie"] = variable
    return salida


def _trimestral_oficial_larga(df, variable):
    salida = df.rename(columns={variable: "valor"})[["REGION", "ANO4", "TRIMESTRE", "valor"]].copy()
    salida["serie"] = variable
    salida["origen"] = cr.ORIGEN_OFICIAL
    return salida


canasta_mensual_larga = pd.concat([
    _mensual_retropolada_larga(cba_regiones_mensual, "cba_adulto_equivalente"),
    _mensual_retropolada_larga(cbt_regiones_mensual, "cbt_adulto_equivalente"),
    _mensual_oficial_larga(oficial_mensual_cba, "cba_adulto_equivalente"),
    _mensual_oficial_larga(oficial_mensual_cbt, "cbt_adulto_equivalente"),
], ignore_index=True)

canasta_trimestral_larga = pd.concat([
    _trimestral_retropolada_larga(canasta_trimestral_retropolada, "cba_adulto_equivalente"),
    _trimestral_retropolada_larga(canasta_trimestral_retropolada, "cbt_adulto_equivalente"),
    _trimestral_oficial_larga(oficial_trimestral, "cba_adulto_equivalente"),
    _trimestral_oficial_larga(oficial_trimestral, "cbt_adulto_equivalente"),
], ignore_index=True)

META.mkdir(parents=True, exist_ok=True)
canasta_mensual_larga.to_csv(META / "canasta_retropolada_mensual.csv", index=False)
canasta_trimestral_larga.to_csv(META / "canasta_retropolada_trimestral.csv", index=False)
print(f"{len(canasta_mensual_larga)} filas mensuales, {len(canasta_trimestral_larga)} filas trimestrales")
print(canasta_mensual_larga["origen"].value_counts())

## Validación a — Consistencia de anclas

¿Cuánto creció la canasta oficial entre las dos anclas, comparado con la inflación
que el índice encadenado le atribuye al mismo lapso? Una brecha grande indicaría que
el supuesto de Engel fijo (misma trayectoria para CBA y CBT) no absorbe toda la
diferencia -- se reporta, no se corrige.

In [ ]:
inflacion_indice_ancla_a_ancla = indice[pd.Timestamp("2016-04-01")] / indice[pd.Timestamp("2006-12-01")] - 1

filas_validacion_anclas = []
for variable in ("cba_adulto_equivalente", "cbt_adulto_equivalente"):
    valor_dic_2006 = ancla_dic_2006[variable]
    valor_abr_2016 = ancla_abr_2016.set_index("REGION").loc[cr.REGION_GBA, variable]
    inflacion_ancla = valor_abr_2016 / valor_dic_2006 - 1
    filas_validacion_anclas.append({
        "variable": variable, "valor_dic_2006": valor_dic_2006, "valor_abr_2016": valor_abr_2016,
        "inflacion_ancla_pct": 100 * inflacion_ancla, "inflacion_indice_pct": 100 * inflacion_indice_ancla_a_ancla,
        "brecha_pp": 100 * (inflacion_ancla - inflacion_indice_ancla_a_ancla),
    })
validacion_anclas = pd.DataFrame(filas_validacion_anclas)
validacion_anclas.to_csv(META / "canasta_retropolada_validacion_anclas.csv", index=False)
display(validacion_anclas)

## Validación b — Continuidad empírica en el empalme

El empalme final es exacto *por construcción* (cubierto por
`test_continuidad_en_ambos_extremos`) -- lo que no está garantizado es que el último
mes retropolado quede cerca del nivel oficial, por región.

In [ ]:
marzo_2016 = canasta_mensual_larga[
    (canasta_mensual_larga["ANO4"] == 2016) & (canasta_mensual_larga["MES"] == 3)
    & (canasta_mensual_larga["origen"] == cr.ORIGEN_DOS_ANCLAS)]
abril_2016 = canasta_mensual_larga[
    (canasta_mensual_larga["ANO4"] == 2016) & (canasta_mensual_larga["MES"] == 4)
    & (canasta_mensual_larga["origen"] == cr.ORIGEN_OFICIAL)]

continuidad = marzo_2016.merge(abril_2016, on=["REGION", "serie"], suffixes=("_marzo_2016", "_abril_2016"))
continuidad["salto_pct"] = 100 * (continuidad["valor_abril_2016"] / continuidad["valor_marzo_2016"] - 1)
continuidad = continuidad[["REGION", "serie", "valor_marzo_2016", "valor_abril_2016", "salto_pct"]]
continuidad.to_csv(META / "canasta_retropolada_continuidad_abr2016.csv", index=False)
display(continuidad.sort_values(["serie", "REGION"]))

## Validación c — Sensibilidad: un ancla vs. dos anclas

Filtra `origen` en la misma tabla mensual -- no recalcula nada.

In [ ]:
una_ancla = canasta_mensual_larga[canasta_mensual_larga["origen"] == cr.ORIGEN_UNA_ANCLA]
dos_anclas = canasta_mensual_larga[canasta_mensual_larga["origen"] == cr.ORIGEN_DOS_ANCLAS]

sensibilidad = una_ancla.merge(dos_anclas, on=["REGION", "ANO4", "MES", "serie"],
                                suffixes=("_1_ancla", "_2_anclas"))
sensibilidad["diferencia_pct"] = 100 * (sensibilidad["valor_1_ancla"] / sensibilidad["valor_2_anclas"] - 1)
sensibilidad = sensibilidad[["REGION", "ANO4", "MES", "serie", "valor_1_ancla", "valor_2_anclas", "diferencia_pct"]]
sensibilidad.to_csv(META / "canasta_retropolada_sensibilidad_anclas.csv", index=False)

resumen_sensibilidad = sensibilidad.groupby("serie")["diferencia_pct"].agg(["min", "mean", "max"])
display(resumen_sensibilidad)
print("diferencia máxima (valor absoluto) por serie:")
display(sensibilidad.loc[sensibilidad["diferencia_pct"].abs().groupby(sensibilidad["serie"]).idxmax()])

## Validación d — Tasas semestrales (personas) vs. Tornarolli/CEDLAS

Pobreza/indigencia propia por semestre histórico, ponderada por `PONDERA × n_miembros`
(no hay `PONDIH` antes de 2016) sobre los hogares con ingreso declarado -- usa la
canasta trimestral de dos anclas recién construida (un semestre con hueco de
publicación usa un semestre móvil vecino). Solo contraste: no se ajusta nada para
acercarse a las series externas.

In [ ]:
canasta_trim_dos_anclas = canasta_trimestral_retropolada[
    canasta_trimestral_retropolada["origen"] == cr.ORIGEN_DOS_ANCLAS]

COLUMNAS_HOGAR_HIST = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "ITF", "ingreso_no_declarado"]
COLUMNAS_IND_HIST = ["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "PONDERA"]


def construir_pobreza_historica(anio, trimestre):
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR_HIST, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_IND_HIST, NUCLEO)
    if hogar is None or individual is None:
        return None
    hogar = pm.agregar_region(hogar, ARMONIZADO, anio, trimestre)
    canasta_t = canasta_trim_dos_anclas[
        (canasta_trim_dos_anclas["ANO4"] == anio) & (canasta_trim_dos_anclas["TRIMESTRE"] == trimestre)]
    resultado = distancia_linea_pobreza.calcular_distancia_linea_pobreza(hogar, individual, canasta_t[["REGION", "cba_adulto_equivalente", "cbt_adulto_equivalente"]])
    peso = ponderadores.peso_hogar_historico(individual)
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    idx = pd.MultiIndex.from_frame(resultado[["CODUSU", "NRO_HOGAR"]])
    resultado["PONDERA"] = peso.reindex(idx).to_numpy()
    resultado["n_miembros"] = n_miembros.reindex(idx).to_numpy()
    return resultado


tasa_ponderada = distancia_linea_pobreza.tasa_ponderada


def tasas_semestre_historico(componentes):
    partes = []
    for anio, trimestre in componentes:
        df = construir_pobreza_historica(anio, trimestre)
        if df is None:
            return None
        partes.append(df)
    df = pd.concat(partes, ignore_index=True)
    df["peso_personas"] = df["PONDERA"] * df["n_miembros"]
    return {
        "pobreza_personas": tasa_ponderada(df, "pobre", "peso_personas"),
        "indigencia_personas": tasa_ponderada(df, "indigente", "peso_personas"),
    }


semestres_historicos_computables = cr.semestres_historicos_computables


SEMESTRES_HIST = semestres_historicos_computables(TRIMESTRES_HISTORICOS)
print(f"{len(SEMESTRES_HIST)} semestres históricos computables")

t0 = time.time()
filas_propias = []
for anio, semestre, componentes, es_movil in SEMESTRES_HIST:
    tasas = tasas_semestre_historico(componentes)
    if tasas is None:
        continue
    filas_propias.append({"ANO4": anio, "SEMESTRE": semestre, "semestre_movil": es_movil, **tasas})
tasas_propias = pd.DataFrame(filas_propias)
tasas_propias.to_csv(META / "canasta_retropolada_tasas_semestrales.csv", index=False)
print(f"{len(tasas_propias)} semestres, {time.time() - t0:.1f}s")
display(tasas_propias)

### Contraste contra CEDLAS/Tornarolli (Anexo B)

CEDLAS reporta personas -- se compara solo `pobreza_personas`/`indigencia_personas`
contra `pobreza_serie_comparable`/`indigencia_serie_comparable`.

In [ ]:
cedlas = pd.read_csv(RETROPOLACION / "cedlas" / "series_comparables_anexo_b.csv")
cedlas = cedlas.rename(columns={c: c.replace("indig_", "indigencia_").replace("pob_", "pobreza_")
                                 for c in cedlas.columns})
# "semestre_movil" también existe en CEDLAS (su propio criterio de semestre móvil) --
# se descarta para evitar que el merge la duplique: la columna que importa es la
# nuestra (ya en tasas_propias), no la de CEDLAS.
cedlas = cedlas.drop(columns=["semestre", "semestre_movil"])

comparacion_cedlas = tasas_propias.merge(cedlas, on=["ANO4", "SEMESTRE"], how="inner")
comparacion_cedlas["brecha_pobreza_pp"] = (
    comparacion_cedlas["pobreza_personas"] - comparacion_cedlas["pobreza_serie_comparable"])
comparacion_cedlas["brecha_indigencia_pp"] = (
    comparacion_cedlas["indigencia_personas"] - comparacion_cedlas["indigencia_serie_comparable"])
comparacion_cedlas.to_csv(META / "canasta_retropolada_comparacion_cedlas.csv", index=False)

print(f"{len(comparacion_cedlas)} semestres comparados (de {len(tasas_propias)} propios)")
print(f"brecha media pobreza: {comparacion_cedlas['brecha_pobreza_pp'].mean():+.2f} pp "
      f"(desvío {comparacion_cedlas['brecha_pobreza_pp'].std():.2f} pp)")
print(f"brecha media indigencia: {comparacion_cedlas['brecha_indigencia_pp'].mean():+.2f} pp "
      f"(desvío {comparacion_cedlas['brecha_indigencia_pp'].std():.2f} pp)")
display(comparacion_cedlas[["ANO4", "SEMESTRE", "semestre_movil", "pobreza_personas", "pobreza_serie_comparable",
                             "brecha_pobreza_pp", "indigencia_personas", "indigencia_serie_comparable",
                             "brecha_indigencia_pp"]])

## Resumen

Las validaciones quedan en `data/meta/canasta_retropolada_*.csv` para referencia.
El resultado fue aceptado -- el siguiente paso, calcular la distancia por hogar (`distancia_linea_pobreza.calcular_distancia_linea_pobreza`)
sobre la era histórica con esta canasta, se construyó en
`05c_distancia_historica.ipynb`.